# FarmFederate — Real Data Colab Training

Full pipeline using **real data from Google Drive**:
- **Text**: `data/{class}/text.csv` (BLIP captions + pvvqa)
- **Images**: `data/{class}/images/` (PlantVillage + Beans)

### What runs
| Section | Content |
|---------|--------|
| [4/8] | 5 LLM models (DistilBERT, BERT-tiny, RoBERTa-tiny, ALBERT-tiny, MobileBERT) |
| [5/8] | 5 ViT models (ViT-Base, DeiT-tiny, Swin-tiny, ConvNeXT-tiny, EfficientNet) |
| [6/8] | 8 VLM fusion architectures (concat, attention, gated, CLIP, Flamingo, BLIP2, CoCa, Unified-IO) |
| [7/8] | Centralized vs Federated comparison (LLM, ViT, VLM) |
| [8/8] | 25+ plots + results JSON saved to Drive |

### Setup
1. **Runtime → Change runtime type → T4 GPU**
2. Upload `data/` to `MyDrive/FarmFederate/data/` (one folder per class, each with `text.csv` + `images/`)
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Clone / update repo ──────────────────────────────────────────────
import os

REPO_URL = 'https://github.com/Solventerritory/FarmFederate-Advisor.git'
REPO_DIR = '/content/FarmFederate'
BRANCH   = 'feature/multimodal-work'

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

print('Repo ready at', REPO_DIR)

In [ ]:
# ── Cell 3: Copy data from Drive → /content/FarmFederate/data ────────────────
import shutil, os

DRIVE_DATA = '/content/drive/MyDrive/FarmFederate/data'
LOCAL_DATA = '/content/FarmFederate/data'

if os.path.exists(DRIVE_DATA):
    shutil.copytree(DRIVE_DATA, LOCAL_DATA, dirs_exist_ok=True)
    print(f'Data copied: {DRIVE_DATA} -> {LOCAL_DATA}')
else:
    print(f'WARNING: {DRIVE_DATA} not found.')
    print('Upload your data/ folder to Drive at MyDrive/FarmFederate/data/')

# Verify
print('\nData summary:')
print(f'{"Class":<15} {"Texts":>8} {"Images":>8}')
print('-' * 33)
for cls in ['water_stress', 'nutrient_def', 'pest_risk', 'disease_risk', 'heat_stress']:
    txt_path = f'{LOCAL_DATA}/{cls}/text.csv'
    img_path = f'{LOCAL_DATA}/{cls}/images'
    txt  = len(open(txt_path).readlines()) - 1 if os.path.exists(txt_path) else 0
    imgs = len([f for f in os.listdir(img_path) if f.endswith(('.jpg', '.png'))]) if os.path.exists(img_path) else 0
    print(f'{cls:<15} {txt:>8} {imgs:>8}')

In [ ]:
# ── Cell 4: Install dependencies ─────────────────────────────────────────────
!pip install -q torch torchvision transformers scikit-learn pandas numpy \
               matplotlib seaborn tqdm pillow faiss-cpu datasets
print('Dependencies installed.')

In [ ]:
# ── Cell 5: GPU check ────────────────────────────────────────────────────────
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU — Runtime → Change runtime type → T4 GPU')

---
## Part A — Classification Training (5 LLM + 5 ViT + 8 VLM + Fed/Cent)

In [ ]:
# ── Cell 6: Configuration ────────────────────────────────────────────────────
import sys
from pathlib import Path
sys.path.insert(0, '/content/FarmFederate/backend')

from FarmFederate_Colab_Complete import Config

config = Config(
    epochs                = 15,
    batch_size            = 16,
    max_samples_per_class = 800,
    fed_rounds            = 5,
    num_clients           = 3,
    learning_rate         = 1e-4,
)

# Save outputs to Drive so they survive disconnects
config.checkpoint_dir = Path('/content/drive/MyDrive/FarmFederate/checkpoints')
config.output_dir     = Path('/content/drive/MyDrive/FarmFederate/outputs')
config.plots_dir      = Path('/content/drive/MyDrive/FarmFederate/plots')

for d in [config.checkpoint_dir, config.output_dir, config.plots_dir]:
    d.mkdir(parents=True, exist_ok=True)

print('Configuration:')
for k in ['epochs', 'batch_size', 'max_samples_per_class', 'fed_rounds', 'num_clients', 'learning_rate']:
    print(f'  {k:<25}: {getattr(config, k)}')
print(f'  {"checkpoint_dir":<25}: {config.checkpoint_dir}')

In [ ]:
# ── Cell 7: Run full training pipeline ───────────────────────────────────────
# Trains 5 LLM + 5 ViT + 8 VLM + centralized/federated comparison
# Checkpoints auto-saved to Drive after each model

from FarmFederate_Colab_Complete import run_training_real_data

results = run_training_real_data(config=config)

In [ ]:
# ── Cell 8: Results summary table ────────────────────────────────────────────
import numpy as np

print('=' * 70)
print('CLASSIFICATION RESULTS SUMMARY')
print('=' * 70)
print(f'\n{"Model":<22} {"F1 Micro":>10} {"F1 Macro":>10} {"Accuracy":>10} {"Params":>12}')
print('-' * 68)

for section, label in [(results.get('llm_models', {}), 'LLM'),
                        (results.get('vit_models', {}), 'ViT'),
                        (results.get('vlm_models', {}), 'VLM')]:
    print(f'\n--- {label} Models ---')
    for name, r in section.items():
        print(f'{name:<22} {r.get("f1",0):>10.4f} {r.get("f1_macro",r.get("f1",0)):>10.4f} '
              f'{r.get("accuracy",0):>10.4f} {r.get("params",0):>12,}')

print('\n--- Centralized vs Federated ---')
print(f'{"Type":<10} {"Centralized F1":>16} {"Federated F1":>14} {"Diff":>8} {"Winner":>12}')
print('-' * 64)
for mt in ['LLM', 'ViT', 'VLM']:
    c = results.get('centralized', {}).get(mt, {}).get('f1', 0)
    f = results.get('federated',   {}).get(mt, {}).get('f1', 0)
    d = f - c
    w = 'Federated' if d > 0 else ('Centralized' if d < 0 else 'Tie')
    print(f'{mt:<10} {c:>16.4f} {f:>14.4f} {d:>+8.4f} {w:>12}')

all_f1 = {}
for g, prefix in [(results.get('llm_models',{}), 'LLM'),
                   (results.get('vit_models',{}), 'ViT'),
                   (results.get('vlm_models',{}), 'VLM')]:
    for n, r in g.items(): all_f1[f'{prefix}-{n}'] = r.get('f1', 0)

best = max(all_f1, key=all_f1.get)
print(f'\nBest model : {best}  F1={all_f1[best]:.4f}')
print(f'LLM avg F1 : {np.mean([r.get("f1",0) for r in results.get("llm_models",{}).values()]):.4f}')
print(f'ViT avg F1 : {np.mean([r.get("f1",0) for r in results.get("vit_models",{}).values()]):.4f}')
print(f'VLM avg F1 : {np.mean([r.get("f1",0) for r in results.get("vlm_models",{}).values()]):.4f}')

In [ ]:
# ── Cell 9: Training curves ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, title, group in zip(
    axes,
    ['LLM Models', 'ViT Models', 'VLM Fusion'],
    [results.get('llm_models', {}), results.get('vit_models', {}), results.get('vlm_models', {})]
):
    for name, r in group.items():
        val_f1 = r.get('history', {}).get('val_f1', [])
        if val_f1:
            ax.plot(val_f1, label=name, marker='o', markersize=3)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Val F1')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/training_curves.png', dpi=150)
plt.show()
print('Saved: training_curves.png')

In [ ]:
# ── Cell 10: F1 bar chart — all 18 models ────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

names, f1s, colors = [], [], []
for n, r in results.get('llm_models', {}).items():
    names.append(n); f1s.append(r.get('f1', 0)); colors.append('#2196F3')
for n, r in results.get('vit_models', {}).items():
    names.append(n); f1s.append(r.get('f1', 0)); colors.append('#4CAF50')
for n, r in results.get('vlm_models', {}).items():
    names.append(n); f1s.append(r.get('f1', 0)); colors.append('#FF5722')

fig, ax = plt.subplots(figsize=(18, 6))
bars = ax.bar(range(len(names)), f1s, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('F1 Score (Micro)')
ax.set_title('FarmFederate — All Models F1 Comparison (Real Data)', fontsize=14, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.axhline(y=np.mean(f1s), color='black', linestyle='--', alpha=0.4)
for bar, val in zip(bars, f1s):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=7)
ax.legend(handles=[
    Patch(color='#2196F3', label='LLM (Text)'),
    Patch(color='#4CAF50', label='ViT (Image)'),
    Patch(color='#FF5722', label='VLM (Multimodal)'),
    plt.Line2D([0],[0], color='black', linestyle='--', label=f'Mean={np.mean(f1s):.3f}'),
])
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/f1_all_models.png', dpi=150)
plt.show()
print('Saved: f1_all_models.png')

In [ ]:
# ── Cell 11: Centralized vs Federated bar chart ───────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

model_types = ['LLM', 'ViT', 'VLM']
cent_f1s = [results.get('centralized', {}).get(mt, {}).get('f1', 0) for mt in model_types]
fed_f1s  = [results.get('federated',   {}).get(mt, {}).get('f1', 0) for mt in model_types]

x, w = np.arange(len(model_types)), 0.35
fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, cent_f1s, w, label='Centralized', color='#1976D2')
b2 = ax.bar(x + w/2, fed_f1s,  w, label='Federated',   color='#388E3C')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(model_types, fontsize=12)
ax.set_ylabel('F1 Score (Micro)')
ax.set_title('Centralized vs Federated Learning', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.05)
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/centralized_vs_federated.png', dpi=150)
plt.show()
print('Saved: centralized_vs_federated.png')

In [ ]:
# ── Cell 12: Per-class F1 heatmap — all models ───────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

STRESS_CLASSES = ['water_stress', 'nutrient_def', 'pest_risk', 'disease_risk', 'heat_stress']
short_cls = [c.replace("_", "\n") for c in STRESS_CLASSES]

all_names, all_perclass = [], []
for mtype, group in [('LLM', results.get('llm_models', {})),
                     ('ViT', results.get('vit_models', {})),
                     ('VLM', results.get('vlm_models', {}))]:
    for name, r in group.items():
        pc = r.get('per_class_f1', r.get('per_class', None))
        if pc is not None:
            all_names.append(f'{mtype}-{name}')
            all_perclass.append(list(pc.values()) if isinstance(pc, dict) else list(pc))

if all_perclass:
    matrix = np.array(all_perclass)
    fig, ax = plt.subplots(figsize=(10, max(5, len(all_names) * 0.45)))
    sns.heatmap(matrix, annot=True, fmt='.2f', cmap='RdYlGn',
                xticklabels=short_cls, yticklabels=all_names,
                vmin=0, vmax=1, linewidths=0.4, ax=ax,
                cbar_kws={'label': 'F1 Score'})
    ax.set_title('Per-Class F1 Score — All Models', fontsize=13, fontweight='bold')
    ax.set_xlabel('Stress Class')
    ax.set_ylabel('Model')
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/FarmFederate/plots/per_class_f1_heatmap.png', dpi=150)
    plt.show()
    print('Saved: per_class_f1_heatmap.png')
else:
    print('per_class_f1 not in results — skipping heatmap')

In [ ]:
# ── Cell 13: Model complexity vs F1 scatter ───────────────────────────────────
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

fig, ax = plt.subplots(figsize=(10, 6))
_colors  = {'LLM': '#2196F3', 'ViT': '#4CAF50', 'VLM': '#FF5722'}
_markers = {'LLM': 'o',       'ViT': 's',        'VLM': '^'}

for mtype, group in [('LLM', results.get('llm_models', {})),
                     ('ViT', results.get('vit_models', {})),
                     ('VLM', results.get('vlm_models', {}))]:
    for name, r in group.items():
        params = r.get('params', 0)
        f1     = r.get('f1', 0)
        if params > 0:
            ax.scatter(params / 1e6, f1,
                       color=_colors[mtype], marker=_markers[mtype],
                       s=110, edgecolors='white', linewidths=0.6, zorder=3)
            ax.annotate(name, (params / 1e6, f1),
                        textcoords='offset points', xytext=(5, 3),
                        fontsize=7, color=_colors[mtype])

ax.legend(handles=[Patch(color=c, label=f'{t} Models') for t, c in _colors.items()], fontsize=10)
ax.set_xlabel('Model Parameters (Millions)', fontsize=11)
ax.set_ylabel('F1 Score (Micro)', fontsize=11)
ax.set_title('Model Complexity vs Classification Accuracy', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/FarmFederate/plots/complexity_vs_accuracy.png', dpi=150)
plt.show()
print('Saved: complexity_vs_accuracy.png')

In [ ]:
# ── Cell 14: Save classification results JSON ─────────────────────────────────
import json, shutil

results_path = '/content/drive/MyDrive/FarmFederate/outputs/complete_results.json'
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2, default=str)
print(f'Results saved : {results_path}')

zip_path = '/content/drive/MyDrive/FarmFederate/farmfederate_checkpoints'
shutil.make_archive(zip_path, 'zip', str(config.checkpoint_dir))
print(f'Checkpoints   : {zip_path}.zip')

---
## Part B — RAG Pipeline (Federated Retrieval-Augmented Generation)

Trains a federated RAG system on the same real data:
- **Encoder**: DistilRoBERTa (text) + CNN (image) → fused 512-d
- **Knowledge base**: Per-farm FAISS index — never transmitted to server
- **Training**: FedAvg with InfoNCE contrastive loss + BCE advisory loss
- **Metrics**: Recall@K, MRR, NDCG@K, per-class F1, embedding drift

| Mode | Farms | FL rounds | Samples/farm | ~Time |
|------|-------|-----------|--------------|-------|
| `quick` | 2 | 2 | 100 | 5 min |
| `standard` | 3 | 5 | 400 | 20 min |
| `full` | 5 | 8 | 800 | 40 min |

In [ ]:
# ── Cell 15: RAG configuration ───────────────────────────────────────────────
import os

RAG_MODE       = 'standard'   # 'quick' | 'standard' | 'full'
GOOGLE_API_KEY = ''           # optional: Gemini key for advisory generation

RAG_OUT_DIR  = '/content/drive/MyDrive/FarmFederate/rag_results'
RAG_CKPT_DIR = '/content/drive/MyDrive/FarmFederate/rag_checkpoints'
os.makedirs(RAG_OUT_DIR,  exist_ok=True)
os.makedirs(RAG_CKPT_DIR, exist_ok=True)

_MODES = {
    'quick':    dict(num_farms=2, fed_rounds=2, local_epochs=1, samples_per_farm=100,  rag_rounds=2,  top_k=3),
    'standard': dict(num_farms=3, fed_rounds=5, local_epochs=2, samples_per_farm=400,  rag_rounds=10, top_k=5),
    'full':     dict(num_farms=5, fed_rounds=8, local_epochs=3, samples_per_farm=800,  rag_rounds=15, top_k=5),
}
cfg = _MODES[RAG_MODE]

print(f'RAG mode       : {RAG_MODE}')
print(f'Farms          : {cfg["num_farms"]}')
print(f'FL rounds      : {cfg["fed_rounds"]}')
print(f'Samples/farm   : {cfg["samples_per_farm"]}')
print(f'Top-K          : {cfg["top_k"]}')
print(f'Output dir     : {RAG_OUT_DIR}')

In [ ]:
# ── Cell 16: Run RAG pipeline ─────────────────────────────────────────────────
import sys, types
sys.path.insert(0, '/content/FarmFederate/backend')

with open('/content/FarmFederate/backend/FarmFederate_RAG_Colab.py', 'r') as _f:
    _src = _f.read()

_src = (_src
    .replace('EXECUTION_MODE = "standard"', f'EXECUTION_MODE = "{RAG_MODE}"')
    .replace('GOOGLE_API_KEY = ""',         f'GOOGLE_API_KEY = "{GOOGLE_API_KEY}"')
)

rag_mod = types.ModuleType('FarmFederate_RAG_Colab')
rag_mod.__file__ = '/content/FarmFederate/backend/FarmFederate_RAG_Colab.py'
sys.modules['FarmFederate_RAG_Colab'] = rag_mod

exec(compile(_src, rag_mod.__file__, 'exec'), rag_mod.__dict__)
rag_results = rag_mod.run()
print('\nRAG pipeline complete.')

In [ ]:
# ── Cell 17: RAG metrics summary ─────────────────────────────────────────────
import json

print('=' * 60)
print('RAG PIPELINE — RESULTS SUMMARY')
print('=' * 60)

if rag_results and isinstance(rag_results, dict):
    ret   = rag_results.get('retrieval_metrics', rag_results.get('ret_metrics', {}))
    cls_m = rag_results.get('classification',    rag_results.get('cls_metrics', {}))
    top_k = cfg['top_k']

    print(f'\n--- Retrieval ---')
    for key in [f'recall_at_{top_k}', 'mrr', f'ndcg_at_{top_k}', 'kb_coverage']:
        val   = ret.get(key, 'N/A')
        label = key.replace('_', ' ').replace('at', '@').upper()
        print(f'  {label:<20}: {val:.4f}' if isinstance(val, float) else f'  {label:<20}: {val}')

    print(f'\n--- Classification ---')
    for key in ['f1_macro', 'f1_micro', 'accuracy']:
        val = cls_m.get(key, 'N/A')
        print(f'  {key:<20}: {val:.4f}' if isinstance(val, float) else f'  {key:<20}: {val}')

    print(f'\n--- Federated ---')
    fed = rag_results.get('federated', {})
    print(f'  Rounds completed    : {fed.get("rounds", cfg["fed_rounds"])}')
    print(f'  Final ret loss      : {fed.get("final_ret_loss", "N/A")}')
    print(f'  Embedding drift     : {rag_results.get("drift", fed.get("drift", "N/A"))}')

    rag_json = f'{RAG_OUT_DIR}/rag_results_summary.json'
    with open(rag_json, 'w') as f:
        json.dump(rag_results, f, indent=2, default=str)
    print(f'\nFull results saved: {rag_json}')
else:
    print('No structured results — check plots in', RAG_OUT_DIR)

In [ ]:
# ── Cell 18: Display all RAG plots ───────────────────────────────────────────
import os, matplotlib.pyplot as plt, matplotlib.image as mpimg

plot_files = sorted(f for f in os.listdir(RAG_OUT_DIR) if f.endswith('.png'))
print(f'Found {len(plot_files)} RAG plots')

if plot_files:
    n_cols = 2
    n_rows = (len(plot_files) + 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 5 * n_rows))
    axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]
    for ax, fname in zip(axes, plot_files):
        img = mpimg.imread(os.path.join(RAG_OUT_DIR, fname))
        ax.imshow(img); ax.axis('off')
        ax.set_title(fname.replace('.png', '').replace('_', ' '), fontsize=10)
    for ax in axes[len(plot_files):]:
        ax.axis('off')
    plt.tight_layout()
    combined = f'{RAG_OUT_DIR}/rag_all_plots.png'
    plt.savefig(combined, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved: {combined}')

---
## Part C — Final Summary & Download

In [ ]:
# ── Cell 19: Combined summary ─────────────────────────────────────────────────
import numpy as np

print('=' * 70)
print('FARMFEDERATE — FULL PIPELINE SUMMARY')
print('=' * 70)

all_f1 = {}
for g, p in [(results.get('llm_models',{}), 'LLM'),
             (results.get('vit_models',{}),  'ViT'),
             (results.get('vlm_models',{}),  'VLM')]:
    for n, r in g.items(): all_f1[f'{p}-{n}'] = r.get('f1', 0)

best = max(all_f1, key=all_f1.get)
print(f'\nClassification (18 models on real data):')
print(f'  Best model   : {best}  F1={all_f1[best]:.4f}')
print(f'  LLM avg F1   : {np.mean([r.get("f1",0) for r in results.get("llm_models",{}).values()]):.4f}')
print(f'  ViT avg F1   : {np.mean([r.get("f1",0) for r in results.get("vit_models",{}).values()]):.4f}')
print(f'  VLM avg F1   : {np.mean([r.get("f1",0) for r in results.get("vlm_models",{}).values()]):.4f}')

print(f'\nFederated vs Centralized:')
for mt in ['LLM', 'ViT', 'VLM']:
    c = results.get('centralized', {}).get(mt, {}).get('f1', 0)
    f = results.get('federated',   {}).get(mt, {}).get('f1', 0)
    print(f'  {mt}: Centralized={c:.4f}  Federated={f:.4f}  ({f-c:+.4f})')

print(f'\nRAG Pipeline ({RAG_MODE} mode):')
if rag_results and isinstance(rag_results, dict):
    ret   = rag_results.get('retrieval_metrics', rag_results.get('ret_metrics', {}))
    top_k = cfg['top_k']
    print(f'  Recall@{top_k}     : {ret.get(f"recall_at_{top_k}", "N/A")}')
    print(f'  MRR          : {ret.get("mrr", "N/A")}')
    print(f'  NDCG@{top_k}       : {ret.get(f"ndcg_at_{top_k}", "N/A")}')

print(f'\nAll outputs on Google Drive:')
print(f'  MyDrive/FarmFederate/outputs/          — classification results JSON')
print(f'  MyDrive/FarmFederate/plots/            — classification plots')
print(f'  MyDrive/FarmFederate/checkpoints/      — model checkpoints')
print(f'  MyDrive/FarmFederate/rag_results/      — RAG plots + metrics')
print(f'  MyDrive/FarmFederate/rag_checkpoints/  — RAG checkpoint')

In [ ]:
# ── Cell 20: Zip checkpoints & (optional) download ───────────────────────────
import shutil

shutil.make_archive(
    '/content/drive/MyDrive/FarmFederate/farmfederate_checkpoints',
    'zip', str(config.checkpoint_dir)
)
shutil.make_archive(
    '/content/drive/MyDrive/FarmFederate/rag_checkpoints_archive',
    'zip', RAG_CKPT_DIR
)
print('Zipped to Drive:')
print('  farmfederate_checkpoints.zip  — classification models')
print('  rag_checkpoints_archive.zip   — RAG model')

# Uncomment to download directly to your machine:
# from google.colab import files
# files.download('/content/drive/MyDrive/FarmFederate/farmfederate_checkpoints.zip')
# files.download('/content/drive/MyDrive/FarmFederate/rag_checkpoints_archive.zip')